# Stage 5 — Strategy–Result Research Design

This notebook extends the completed portfolio-performance track with a separately governed strategy-and-company-results track. It freezes the research questions, time scope, strategic-lever taxonomy, outcome taxonomy, reporting-entity perimeter, source hierarchy, attribution rules, and linkage-eligibility gates before new evidence is acquired.

Stage 0–4 analytical outputs remain unchanged and serve only as validated portfolio-performance outcomes. This notebook does not estimate strategy effects, create a composite score, identify an overall winner, or prepare the analytical report.


## Environment Setup

Import the required libraries and define the locked repository commit, required inherited inputs, expected checksums, and deterministic output locations.


In [1]:
from __future__ import annotations

import hashlib
import os
import shutil
import urllib.error
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd

REPOSITORY_FULL_NAME = "Ronaldo-spec/indonesia-fmcg-brand-portfolio-analysis"
INPUT_COMMIT = "888c42ff5a9bab3b8d86eef5c5026eb3203b20bc"

INPUT_LOCKS = {
    "metadata/ownership_registry.csv": "a30edb56b67f77857dfdf2332f3b19409db65a7d3771f7edb3af316b30b34744",
    "metadata/source_registry.csv": "cb19e2efbd46eddebe7d3a22bd1ced13841920679709d93775302bc6ad76594e",
    "metadata/comparability_rules.csv": "2e7c0659f2d7addbe41d2028fd88452a217c6bd0b49623fca70c96cb6b466a97",
    "metadata/stage3_metric_eligibility_rules.csv": "b67fd93e078eff382e47ed45e87f0856f191bb63dea3a839d9be6a58e412e8f9",
    "data/analytical/stage4_final_findings.csv": "847d17369b60f950495467a9ecedbbbd2bd5397d49aa6a845be6dd784b8ee35c",
    "data/analytical/overall_winner_defensibility.csv": "5d055108bd47a9b8fc6e3afe466037270db73b7be05bc861a0489b28696d5765",
    "metadata/stage4c_analysis_validation.csv": "9af33f0c985da44cc40f1f9085a7b18ebdb3c3b5de5a9784fac90102704b844b",
}

OUTPUT_ROOT = Path(os.environ.get("FMCG_STAGE5_OUTPUT_ROOT", "/content/fmcg_stage5_outputs"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", None)

print(f"Locked input commit: {INPUT_COMMIT}")
print(f"Required inherited inputs: {len(INPUT_LOCKS)}")
print(f"Output root: {OUTPUT_ROOT}")


Locked input commit: 888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
Required inherited inputs: 7
Output root: /content/fmcg_stage5_outputs


## Locked Committed Input Retrieval

Retrieve only the inherited metadata and validated Stage 4 evidence required to govern the new research track. Local validation can use an explicitly configured input root.


In [2]:
configured_root = os.environ.get("FMCG_STAGE5_INPUT_ROOT")

if configured_root:
    INPUT_ROOT = Path(configured_root)
    input_mode = "local_validation_root"
else:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError(
            "Run this notebook in Google Colab or set FMCG_STAGE5_INPUT_ROOT "
            "for local validation."
        ) from exc

    github_token = userdata.get("GITHUB_TOKEN")
    if not github_token:
        raise RuntimeError(
            "The Colab Secret GITHUB_TOKEN is unavailable or access has not "
            "been granted to this notebook."
        )

    INPUT_ROOT = Path("/content/fmcg_stage5_inputs")
    if INPUT_ROOT.exists():
        shutil.rmtree(INPUT_ROOT)
    INPUT_ROOT.mkdir(parents=True, exist_ok=True)

    for relative_path in INPUT_LOCKS:
        encoded_path = urllib.parse.quote(relative_path, safe="/")
        api_url = (
            f"https://api.github.com/repos/{REPOSITORY_FULL_NAME}/contents/"
            f"{encoded_path}?ref={INPUT_COMMIT}"
        )
        request = urllib.request.Request(
            api_url,
            headers={
                "Authorization": f"Bearer {github_token}",
                "Accept": "application/vnd.github.raw+json",
                "X-GitHub-Api-Version": "2022-11-28",
                "User-Agent": "fmcg-stage5-colab",
            },
        )

        destination = INPUT_ROOT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)

        try:
            with urllib.request.urlopen(request, timeout=60) as response:
                destination.write_bytes(response.read())
        except urllib.error.HTTPError as exc:
            raise RuntimeError(
                f"GitHub input retrieval failed for {relative_path} "
                f"with HTTP {exc.code}."
            ) from exc

    del github_token
    input_mode = "locked_github_commit"

missing_inputs = [
    relative_path
    for relative_path in INPUT_LOCKS
    if not (INPUT_ROOT / relative_path).exists()
]

if missing_inputs:
    raise FileNotFoundError(f"Missing required inputs: {missing_inputs}")

print(f"Input mode: {input_mode}")
print(f"Required files found: {len(INPUT_LOCKS)}/{len(INPUT_LOCKS)}")


Input mode: locked_github_commit
Required files found: 7/7


## Inherited Input Integrity and Prior-Stage Gate

Verify every inherited file against its locked SHA-256 value and confirm that Stage 4 remains complete with caveats, six registered findings, and a not-defensible overall-winner conclusion.


In [3]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


input_lock_rows = []
for relative_path, expected_sha256 in INPUT_LOCKS.items():
    actual_sha256 = sha256_file(INPUT_ROOT / relative_path)
    input_lock_rows.append(
        {
            "file_path": relative_path,
            "expected_sha256": expected_sha256,
            "actual_sha256": actual_sha256,
            "hash_match": actual_sha256 == expected_sha256,
            "locked_repository_commit": INPUT_COMMIT,
        }
    )

stage5_input_lock = pd.DataFrame(input_lock_rows)
if not stage5_input_lock["hash_match"].all():
    failures = stage5_input_lock.loc[~stage5_input_lock["hash_match"], "file_path"].tolist()
    raise RuntimeError(f"Inherited input checksum failure: {failures}")

ownership = pd.read_csv(INPUT_ROOT / "metadata/ownership_registry.csv", dtype=str, keep_default_na=False)
source_registry = pd.read_csv(INPUT_ROOT / "metadata/source_registry.csv", dtype=str, keep_default_na=False)
comparability_rules = pd.read_csv(INPUT_ROOT / "metadata/comparability_rules.csv", dtype=str, keep_default_na=False)
stage3_rules = pd.read_csv(INPUT_ROOT / "metadata/stage3_metric_eligibility_rules.csv", dtype=str, keep_default_na=False)
stage4_findings = pd.read_csv(INPUT_ROOT / "data/analytical/stage4_final_findings.csv", dtype=str, keep_default_na=False)
winner_gates = pd.read_csv(INPUT_ROOT / "data/analytical/overall_winner_defensibility.csv", dtype=str, keep_default_na=False)
stage4c_validation = pd.read_csv(INPUT_ROOT / "metadata/stage4c_analysis_validation.csv", dtype=str, keep_default_na=False)

expected_groups = {"Wings Group", "Indofood", "Mayora", "Unilever Indonesia"}
if set(ownership["group"].unique()) != expected_groups:
    raise RuntimeError("The inherited ownership registry does not contain exactly the four focal groups.")
if set(stage4_findings["finding_id"]) != {f"FND4_{index:02d}" for index in range(1, 7)}:
    raise RuntimeError("The inherited Stage 4 finding registry is incomplete or altered.")
if winner_gates.loc[winner_gates["gate_id"] == "OWG11", "status"].squeeze() != "not_defensible":
    raise RuntimeError("The inherited overall-winner conclusion is not the validated not-defensible status.")
if stage4c_validation.loc[stage4c_validation["check_id"] == "S4C016", "result"].squeeze() != "PASS_WITH_CAVEAT":
    raise RuntimeError("The inherited Stage 4C final gate is not PASS_WITH_CAVEAT.")

print(f"Input checksums passed: {stage5_input_lock['hash_match'].sum()}/{len(stage5_input_lock)}")
print(f"Ownership records inherited: {len(ownership)}")
print(f"Stage 4 findings inherited: {len(stage4_findings)}")
print("Stage 4 overall-winner conclusion: not_defensible")


Input checksums passed: 7/7
Ownership records inherited: 157
Stage 4 findings inherited: 6
Stage 4 overall-winner conclusion: not_defensible


## Strategy–Result Research Questions

Freeze the questions that will govern evidence acquisition and analysis. The questions distinguish documented strategic actions, intended mechanisms, observed outcomes, attribution strength, and cross-company comparability.


In [4]:
stage5_research_questions = pd.DataFrame(
    [
        ("SRQ01", "documented_strategy", "Which material portfolio, pricing, marketing, distribution, supply, manufacturing, capital-allocation, cost-productivity, and ownership actions were documented for each focal group during the eligible period?", "primary"),
        ("SRQ02", "intended_mechanism", "What mechanism was each documented action intended to influence, and at which brand, category, segment, entity, geography, and period?", "primary"),
        ("SRQ03", "portfolio_product", "How do documented portfolio and product actions align with inherited breadth, focal-category leadership, stability, momentum, and persistence outcomes?", "primary"),
        ("SRQ04", "pricing_pack", "How do documented pricing and pack-architecture actions align with revenue, volume, price/mix, profitability, and eligible brand outcomes?", "primary"),
        ("SRQ05", "marketing_brand", "How do documented marketing and brand-investment actions align with eligible brand or category outcomes without treating temporal association as causation?", "primary"),
        ("SRQ06", "distribution_supply_capacity", "How do documented distribution, supply, procurement, and manufacturing-capacity actions align with availability, volume, sales, and category outcomes?", "primary"),
        ("SRQ07", "capital_cost_productivity", "How do documented capital-allocation and cost-productivity actions align with capacity, growth, profit, margin, and cash-flow outcomes?", "primary"),
        ("SRQ08", "reporting_perimeter", "Which reported outcomes are attributable to an Indonesian brand, category, segment, or company, and which remain consolidated-group or mixed-geography context only?", "primary"),
        ("SRQ09", "alternative_explanations", "Do timing, ownership changes, input costs, exchange rates, inflation, regulation, portfolio changes, or other disclosed factors limit a strategy–result interpretation?", "primary"),
        ("SRQ10", "cross_group_synthesis", "Which strategy–result patterns are directly comparable across focal groups, and which support only within-company or context-specific conclusions?", "primary"),
        ("SRQ11", "overall_conclusion", "Does the combined evidence change any dimension-level conclusion or the not-defensible overall-winner status without a pre-specified comparable scale and weighting framework?", "governance"),
    ],
    columns=["question_id", "question_domain", "research_question", "analytical_role"],
)

if stage5_research_questions["question_id"].duplicated().any():
    raise RuntimeError("Research question IDs must be unique.")

print(f"Frozen Stage 5 research questions: {len(stage5_research_questions)}")


Frozen Stage 5 research questions: 11


## Period and Temporal Alignment Policy

Define the full-year outcome window, the limited pre-window strategy allowance, the separate treatment of the 2026 brand snapshot, and the handling of incomplete-year evidence.


In [5]:
stage5_period_scope = pd.DataFrame(
    [
        ("PER01", "strategy_pre_window", "2021", "eligible_with_caveat", "A 2021 action may be linked only when a documented or pre-specified lag plausibly affects a 2022–2025 outcome."),
        ("PER02", "primary_strategy_window", "2022–2025", "primary", "Use dated or period-bounded documented strategic actions."),
        ("PER03", "primary_corporate_outcome_window", "FY2022–FY2025", "primary", "Use completed annual periods and preserve fiscal-period definitions."),
        ("PER04", "inherited_historical_brand_outcomes", "2022–2025", "primary_with_inherited_caveat", "Retain the validated historical Top Brand methodology cluster and its existing caveats."),
        ("PER05", "documented_current_brand_snapshot", "2026", "separate_current_snapshot", "Retain the documented 2026 focal-category snapshot without bridging it to the historical cluster."),
        ("PER06", "incomplete_2026_corporate_evidence", "2026 interim or year-to-date", "context_only_until_full_year", "Do not compare partial-year results with completed annual periods unless a like-for-like interim comparison is explicitly defined."),
        ("PER07", "outside_window", "Before 2021 or after 2026", "context_only", "Use only when necessary to explain ownership, long-lived capacity, or a clearly documented prior condition."),
    ],
    columns=["period_rule_id", "evidence_role", "period", "eligibility", "required_treatment"],
)

print(f"Temporal-scope rules: {len(stage5_period_scope)}")


Temporal-scope rules: 7


## Strategic-Lever Taxonomy

Classify company actions as strategic levers. Outcomes such as revenue, volume, profit, and margin are deliberately excluded from this taxonomy.


In [6]:
stage5_strategy_taxonomy = pd.DataFrame(
    [
        ("STR01", "portfolio_product", "Portfolio and product strategy", "Launches, reformulation, premiumisation, value-tier development, category entry or exit, and portfolio rationalisation.", "Do not infer an action from a later performance change."),
        ("STR02", "pricing_pack_architecture", "Pricing and pack architecture", "Documented price changes, price points, pack sizes, affordability architecture, premium pricing, and trade terms where disclosed.", "Revenue growth alone is not evidence of a pricing action."),
        ("STR03", "marketing_brand_investment", "Marketing and brand investment", "Documented advertising, promotion, activation, sponsorship, digital marketing, and positioning initiatives.", "Do not treat visibility, awards, or TBI movement as proof of marketing spend or effectiveness."),
        ("STR04", "distribution_channel", "Distribution and channel strategy", "Documented distributor expansion, general-trade, modern-trade, e-commerce, route-to-market, or geographic-access actions.", "Distribution disclosure is not consumer reach unless the metric explicitly measures reach."),
        ("STR05", "supply_procurement", "Supply and procurement strategy", "Documented sourcing, supplier, inventory, logistics, and input-risk actions.", "Do not infer supply strategy solely from margin movement."),
        ("STR06", "manufacturing_capacity", "Manufacturing and capacity strategy", "Documented factory, line, automation, utilisation, quality, or capacity-expansion actions.", "Announced capacity is not realised output unless completion and operation are documented."),
        ("STR07", "capital_allocation", "Capital allocation", "Documented capital expenditure, investment priorities, financing, acquisitions, and divestments.", "Capex amount is not an outcome score and must retain project and entity scope."),
        ("STR08", "cost_productivity", "Cost and productivity management", "Documented productivity, efficiency, restructuring, cost-saving, and working-capital actions.", "Margin improvement is an outcome, not standalone proof of a cost action."),
        ("STR09", "ownership_partnership", "Ownership, partnership, and portfolio-control actions", "Documented acquisition, divestment, joint venture, licensing, and control changes affecting the eligible portfolio.", "Do not give strict-control credit to joint ventures, affiliates, or distribution-only relationships."),
    ],
    columns=["strategy_id", "strategy_code", "strategy_dimension", "included_actions", "prohibited_inference"],
)

if stage5_strategy_taxonomy["strategy_code"].duplicated().any():
    raise RuntimeError("Strategy codes must be unique.")

print(f"Strategic-lever dimensions: {len(stage5_strategy_taxonomy)}")


Strategic-lever dimensions: 9


## Outcome Taxonomy

Separate inherited portfolio outcomes from newly acquired commercial, profitability, operational, and capital outcomes. Every metric retains its source-native definition, unit, entity, geography, and period.


In [7]:
stage5_outcome_taxonomy = pd.DataFrame(
    [
        ("OUT01", "portfolio", "structural_brand_breadth", "count", "inherited_stage4", "eligible_with_caveat", "Brand-family breadth only; not structural category breadth or competitive strength."),
        ("OUT02", "portfolio", "focal_category_leadership", "event_count_or_rate", "inherited_stage4", "eligible_with_caveat", "Selected complete focal-category evidence; not market share or full-market leadership."),
        ("OUT03", "portfolio", "observed_stability", "block_leader_event", "inherited_stage4", "eligible_with_caveat", "Stability is not strength."),
        ("OUT04", "portfolio", "momentum", "block_leader_event", "inherited_stage4", "eligible_with_caveat", "No cross-category magnitude averaging and no 2025–2026 methodology bridge."),
        ("OUT05", "portfolio", "competitive_persistence", "role_specific_event", "inherited_stage4", "descriptive_only", "Incumbent and challenger opportunities are unequal."),
        ("OUT06", "portfolio", "consumer_reach", "source_native_reach_metric", "new_or_existing_source", "not_eligible_currently", "Require a comparable source-native universe; CRP is not market share."),
        ("OUT07", "commercial", "net_sales_or_revenue", "reported_currency", "new_corporate_evidence", "conditional", "Preserve entity, segment, geography, consolidation, and continuing-operation scope."),
        ("OUT08", "commercial", "sales_growth", "percent", "new_corporate_evidence", "conditional", "Separate reported, constant-currency, organic, and derived growth definitions."),
        ("OUT09", "commercial", "volume", "source_native_volume_unit", "new_corporate_evidence", "conditional", "Do not infer volume from revenue without a disclosed bridge."),
        ("OUT10", "commercial", "price_mix", "percent_or_reported_effect", "new_corporate_evidence", "conditional", "Use only an explicitly reported price, mix, or combined price/mix measure."),
        ("OUT11", "profitability", "gross_profit", "reported_currency", "new_corporate_evidence", "conditional", "Require a compatible accounting and entity scope."),
        ("OUT12", "profitability", "gross_margin", "percent", "new_corporate_evidence", "conditional", "Margin is an outcome, not a strategy, and must retain the reported calculation basis."),
        ("OUT13", "profitability", "operating_profit", "reported_currency", "new_corporate_evidence", "conditional", "Separate recurring, underlying, and reported measures."),
        ("OUT14", "profitability", "operating_margin", "percent", "new_corporate_evidence", "conditional", "Do not compare margins across incompatible entities or accounting definitions."),
        ("OUT15", "capital", "capital_expenditure", "reported_currency", "new_corporate_evidence", "conditional", "Capex is an investment input unless an operational result is separately observed."),
        ("OUT16", "operational", "production_capacity_or_utilisation", "source_native_unit", "new_corporate_evidence", "conditional", "Separate announced, installed, commissioned, and utilised capacity."),
        ("OUT17", "operational", "distribution_or_availability", "source_native_unit", "new_corporate_evidence", "conditional", "Do not relabel outlet, distributor, or geographic counts as consumer reach."),
        ("OUT18", "market", "source_defined_market_share", "source_native_percent", "new_external_or_company_evidence", "conditional", "Use the term market share only when the source explicitly measures market share for a defined market."),
    ],
    columns=["outcome_id", "outcome_family", "outcome_metric", "unit_family", "evidence_origin", "current_eligibility", "required_treatment"],
)

if "margin" in set(stage5_strategy_taxonomy["strategy_code"]):
    raise RuntimeError("Margin must remain an outcome rather than a strategic lever.")

print(f"Outcome definitions: {len(stage5_outcome_taxonomy)}")


Outcome definitions: 18


## Reporting-Entity and Portfolio Perimeter

Extend the inherited ownership rules to corporate-result attribution without changing strict-control brand ownership or the frozen Stage 4 performance universe.


In [8]:
stage5_reporting_entity_scope = pd.DataFrame(
    [
        ("ENT01", "Wings Group", "Wings Group strict-control consumer portfolio", "primary_portfolio", "group_or_resolved_operating_entity", "Group attribution is acceptable; exact operating entity must be resolved when a corporate result requires it.", "Do not treat unavailable private-company disclosure as zero or weak performance."),
        ("ENT02", "Wings Group", "PT Lion Wings; PT Glico Wings; PT Calbee Wings and other explicit joint ventures", "non_primary", "joint_venture", "Retain separately and do not give full strict-control credit.", "JV results cannot be merged into the primary Wings portfolio without ownership-proportion and scope support."),
        ("ENT03", "Indofood", "Consumer Branded Products", "primary_portfolio", "segment_or_controlled_entity", "Use brand, category, segment, or controlled-entity outcomes when the reporting perimeter is explicit.", "Do not attribute total Indofood consolidated outcomes to packaged FMCG brands."),
        ("ENT04", "Indofood", "Bogasari consumer flour portfolio", "primary_portfolio", "segment", "Include consumer-facing packaged flour results when defensibly attributable.", "Exclude industrial-only or unrelated segment outcomes from brand-performance interpretation."),
        ("ENT05", "Indofood", "Agribusiness consumer edible oils and fats", "primary_portfolio", "segment", "Include only consumer edible-oil and fat outcomes with adequate product-scope support.", "Do not use upstream plantation results as direct packaged-FMCG outcomes."),
        ("ENT06", "Indofood", "Distribution and upstream-only operations", "context_only", "segment_or_group", "Use only to explain route-to-market or input context when explicitly linked.", "Do not award portfolio-performance credit."),
        ("ENT07", "Mayora", "PT Mayora Indah Tbk and consolidated subsidiaries", "primary_portfolio", "consolidated_or_segment", "Preserve domestic/export and segment scope where disclosed.", "Do not assume consolidated or export-heavy outcomes represent Indonesian household demand."),
        ("ENT08", "Mayora", "PT Tirta Fresindo Jaya / Le Minerale", "sensitivity_only", "related_group_affiliate", "Retain only in the pre-specified extended-group sensitivity.", "Do not include in the strict-control primary Mayora portfolio."),
        ("ENT09", "Unilever Indonesia", "PT Unilever Indonesia Tbk controlled portfolio", "primary_time_varying", "company_brand_or_category", "Credit outcomes only while the brand or business was controlled during the observation period.", "Do not carry disposed businesses into later periods."),
        ("ENT10", "Unilever Indonesia", "Foodservice-only Knorr, Hellmann’s, and Lipton evidence", "context_only", "brand_or_channel", "Use only as channel context outside the primary household-packaged-FMCG universe.", "Do not merge foodservice outcomes into the primary portfolio analysis."),
    ],
    columns=["entity_scope_id", "canonical_group", "reporting_perimeter", "portfolio_role", "preferred_attribution_unit", "required_treatment", "prohibited_use"],
)

if set(stage5_reporting_entity_scope["canonical_group"]) != expected_groups:
    raise RuntimeError("Reporting-entity scope must cover all four focal groups.")

print(f"Reporting-perimeter rules: {len(stage5_reporting_entity_scope)}")


Reporting-perimeter rules: 10


## Source Hierarchy and Acquisition Requirements

Prioritise authoritative, accessible, and legally usable sources. Company-reported strategy and performance statements remain explicitly labelled and are not silently converted into independent conclusions.


In [9]:
stage5_source_requirements = pd.DataFrame(
    [
        ("SRC5_01", 1, "audited_financial_statement", "company_or_regulator", "Reported financial outcomes, accounting scope, notes, and segment information.", "company_reported_or_audited", "Preserve accounting definitions and audit status; do not infer brand-level attribution."),
        ("SRC5_02", 1, "annual_report", "company_or_regulator", "Documented strategy, risk factors, segment results, capex, portfolio actions, and management interpretation.", "company_reported", "Separate factual disclosures from management interpretation."),
        ("SRC5_03", 1, "regulatory_filing", "financial_market_regulator_or_exchange", "Legal-entity events, audited results, acquisitions, divestments, and material disclosures.", "authoritative_filing", "Retain filing date, effective date, entity, and transaction scope."),
        ("SRC5_04", 1, "official_investor_presentation", "company", "Strategy, segment performance, price/mix, volume, capacity, and outlook where disclosed.", "company_reported", "Do not treat guidance or outlook as an actual result."),
        ("SRC5_05", 2, "official_earnings_release", "company", "Period results and management explanations.", "company_reported", "Reconcile headline measures with audited or filed results where possible."),
        ("SRC5_06", 2, "official_corporate_or_brand_announcement", "company_or_brand_owner", "Dated product, pricing, marketing, distribution, capacity, and partnership actions.", "company_reported_action", "Do not infer effectiveness from the announcement."),
        ("SRC5_07", 2, "official_corporate_webpage", "company", "Current portfolio, operating scope, strategy description, and facilities.", "company_reported_current_context", "Do not backdate a current page without archived or dated evidence."),
        ("SRC5_08", 2, "government_or_official_statistics", "government_or_public_authority", "Inflation, input costs, production, trade, regulation, and macro context.", "independent_context", "Match geography, period, category, and unit before comparison."),
        ("SRC5_09", 3, "licensed_or_public_industry_research", "research_provider", "Defined category, channel, consumer, or market measurements.", "external_measurement", "Audit methodology, access rights, redistribution, and metric semantics."),
        ("SRC5_10", 3, "reputable_news_or_interview", "news_publisher", "Corroborating chronology, executive statements, and context not available in primary documents.", "secondary_context", "Do not use as the sole basis for a material quantitative result when primary evidence is available."),
        ("SRC5_11", 4, "undocumented_or_promotional_claim", "unspecified", "Lead for further verification only.", "not_eligible", "Do not use unsupported promotional claims, social posts, or unattributed summaries as final evidence."),
    ],
    columns=["source_requirement_id", "priority_tier", "source_type", "preferred_publisher", "allowed_analytical_role", "claim_label", "required_treatment"],
)

print(f"Source requirement classes: {len(stage5_source_requirements)}")


Source requirement classes: 11


## Attribution and Evidence-Strength Rules

Classify how closely a result matches the strategy’s entity, geography, brand or category, and period. Evidence status records support strength, not causal certainty.


In [10]:
stage5_attribution_evidence_rules = pd.DataFrame(
    [
        ("ATTR01", "brand_category_direct", "The action and result match the same controlled brand or exact category, geography, and eligible period.", "directly_supported", "May support a direct descriptive link; causation still requires stronger identification."),
        ("ATTR02", "business_segment_direct", "The action and result match the same disclosed business segment and eligible period, but brand-level attribution is unavailable.", "supported_with_caveat", "Report at segment level only."),
        ("ATTR03", "indonesia_company_direct", "The result applies to the Indonesian operating company but combines multiple categories or brands.", "supported_with_caveat", "Do not assign the result to an individual brand or category."),
        ("ATTR04", "consolidated_group_only", "The result combines multiple entities, markets, or businesses beyond the primary analytical universe.", "context_only", "Do not treat the result as Indonesian packaged-FMCG performance."),
        ("ATTR05", "mixed_geography", "Domestic and export or international results cannot be separated.", "context_only", "Do not interpret the measure as Indonesian household demand."),
        ("ATTR06", "temporally_aligned_only", "A documented action precedes an outcome, but mechanism, entity, category, or alternative explanations remain insufficiently resolved.", "temporally_aligned", "Describe alignment only; do not state that the action produced the result."),
        ("ATTR07", "company_claimed_link", "The company explicitly attributes a result to a strategy or operating factor.", "company_reported", "Label the attribution as company-reported and seek independent or accounting support."),
        ("ATTR08", "plausible_unverified", "A relationship is plausible but lacks adequate direct, temporal, or scope evidence.", "plausible_but_unverified", "Retain as interpretation or hypothesis, not a finding."),
        ("ATTR09", "not_attributable", "The action and outcome cannot be matched on entity, geography, category, ownership, or period.", "not_assessable", "Do not create a strategy–result link."),
    ],
    columns=["attribution_rule_id", "attribution_class", "definition", "default_evidence_status", "allowed_claim"],
)

allowed_statuses = {
    "directly_supported",
    "supported_with_caveat",
    "temporally_aligned",
    "company_reported",
    "plausible_but_unverified",
    "context_only",
    "not_assessable",
}
if not set(stage5_attribution_evidence_rules["default_evidence_status"]).issubset(allowed_statuses):
    raise RuntimeError("Unexpected evidence-strength status.")

print(f"Attribution and evidence-strength rules: {len(stage5_attribution_evidence_rules)}")


Attribution and evidence-strength rules: 9


## Strategy–Result Linkage Eligibility Gates

Require pre-specified checks before a strategy event can be connected to an outcome. Missing evidence remains unavailable, and cross-company comparison requires a separate comparability decision.


In [11]:
stage5_linkage_eligibility_rules = pd.DataFrame(
    [
        ("LNK01", "documented_action", "A dated or period-bounded strategic action is supported by an eligible source.", "mandatory", "not_eligible"),
        ("LNK02", "ownership_validity", "The focal group controls the relevant brand, business, or result perimeter during the observation period.", "mandatory", "not_eligible"),
        ("LNK03", "entity_alignment", "Strategy and outcome share a defensible entity or segment attribution.", "mandatory_for_direct_link", "context_only"),
        ("LNK04", "geography_alignment", "Strategy and outcome share Indonesia scope or a separately identified geography.", "mandatory_for_indonesia_claim", "context_only"),
        ("LNK05", "brand_category_alignment", "Brand or category scope matches when a brand- or category-level claim is proposed.", "mandatory_for_brand_category_claim", "context_only"),
        ("LNK06", "temporal_order", "The strategy action precedes or overlaps the eligible outcome period in a plausible order.", "mandatory", "not_assessable"),
        ("LNK07", "lag_rationale", "Any lag is pre-specified or documented and not selected after observing the outcome.", "mandatory_when_lagged", "not_assessable"),
        ("LNK08", "outcome_definition", "Metric, unit, period, entity, geography, and source-native semantics are preserved.", "mandatory", "not_eligible"),
        ("LNK09", "baseline_or_comparator", "A compatible prior period, target, comparator, or explicit no-baseline limitation is recorded.", "mandatory_for_change_claim", "descriptive_level_only"),
        ("LNK10", "alternative_factors", "Material disclosed cost, currency, demand, regulation, ownership, and portfolio factors are recorded.", "mandatory_for_interpretation", "supported_with_caveat_at_best"),
        ("LNK11", "company_claim_label", "Company-attributed explanations remain labelled company-reported.", "mandatory", "not_eligible_for_independent_claim"),
        ("LNK12", "missingness", "Unavailable or undisclosed values remain missing rather than zero.", "mandatory", "not_eligible"),
        ("LNK13", "cross_company_comparability", "Metric definition, entity scope, geography, period, unit, and accounting basis are compatible.", "mandatory_for_cross_group_ranking", "within_company_only"),
        ("LNK14", "private_disclosure_neutrality", "Lower public disclosure coverage is not interpreted as weaker strategy or performance.", "mandatory", "not_eligible"),
        ("LNK15", "causal_claim_boundary", "Descriptive alignment is not upgraded to causation without a defensible identification design.", "mandatory", "descriptive_or_associational_only"),
        ("LNK16", "no_composite_upgrade", "Strategy evidence is not converted into an arbitrary score, weighting, or overall winner.", "mandatory", "dimension_or_case_level_only"),
    ],
    columns=["linkage_rule_id", "eligibility_gate", "requirement", "gate_type", "treatment_if_not_met"],
)

if stage5_linkage_eligibility_rules["linkage_rule_id"].duplicated().any():
    raise RuntimeError("Linkage rule IDs must be unique.")

print(f"Strategy–result linkage gates: {len(stage5_linkage_eligibility_rules)}")


Strategy–result linkage gates: 16


## Stage 5 Design Validation

Validate inherited evidence integrity, scope completeness, semantic protections, attribution rules, source hierarchy, period treatment, and the absence of premature analysis or reporting.


In [12]:
validation_rows = []


def add_check(check_id, area, description, result, status="passed", critical_failure="no", required_treatment=""):
    validation_rows.append(
        {
            "check_id": check_id,
            "validation_area": area,
            "check_description": description,
            "result": result,
            "status": status,
            "critical_failure": critical_failure,
            "required_treatment": required_treatment,
        }
    )


add_check("S5D001", "input_integrity", "All inherited Stage 0–4 inputs match their locked SHA-256 values.", f"{stage5_input_lock['hash_match'].sum()}/{len(stage5_input_lock)} inputs passed", required_treatment="Stop Stage 5 if any inherited input differs.")
add_check("S5D002", "prior_stage_gate", "Stage 4C remains PASS_WITH_CAVEAT with no critical failure.", "PASS_WITH_CAVEAT", "passed_with_caveat", required_treatment="Carry all Stage 4 caveats into the strategy–result track.")
add_check("S5D003", "focal_group_scope", "The reporting-entity perimeter covers exactly the four focal groups.", "4/4 focal groups", required_treatment="Do not add or remove focal groups after observing results.")
add_check("S5D004", "stage4_findings", "All six Stage 4 findings remain inherited without modification.", "FND4_01–FND4_06", required_treatment="Treat Stage 4 as the frozen portfolio-outcome module.")
add_check("S5D005", "overall_winner", "The inherited overall-winner conclusion remains not_defensible.", "not_defensible", "passed_with_caveat", required_treatment="Do not use strategy evidence to force one overall winner.")
add_check("S5D006", "research_questions", "Research questions cover actions, mechanisms, outcomes, attribution, alternative factors, and comparability.", f"{len(stage5_research_questions)} questions", required_treatment="Acquire only evidence relevant to the frozen questions.")
add_check("S5D007", "strategy_taxonomy", "Strategic levers are defined before acquisition.", f"{len(stage5_strategy_taxonomy)} strategic-lever dimensions", required_treatment="Do not create post-result strategy categories to favour a group.")
add_check("S5D008", "outcome_taxonomy", "Portfolio, commercial, profitability, capital, operational, and market outcomes remain separated.", f"{len(stage5_outcome_taxonomy)} outcome definitions", required_treatment="Preserve source-native units and definitions.")
add_check("S5D009", "margin_semantics", "Margin is classified as an outcome rather than a strategy.", "gross_margin and operating_margin are outcome metrics", required_treatment="Require a separately documented cost, pricing, mix, or productivity action.")
add_check("S5D010", "market_share_semantics", "Market share is permitted only when the source explicitly measures a defined market share.", "source-defined market share only", required_treatment="Do not relabel TBI, CRP, rank, reach, or revenue as market share.")
add_check("S5D011", "period_scope", "Completed FY2022–FY2025 corporate outcomes are primary; 2026 incomplete-year evidence is separated.", "Primary FY2022–FY2025; 2026 interim context only", "passed_with_caveat", required_treatment="Use like-for-like interim comparisons only when explicitly defined.")
add_check("S5D012", "methodology_boundary", "The 2026 documented brand snapshot is not bridged to the 2022–2025 historical methodology cluster.", "No 2025–2026 longitudinal bridge", required_treatment="Retain inherited methodology clusters.")
add_check("S5D013", "entity_attribution", "Corporate results require explicit brand, category, segment, company, or consolidated attribution.", f"{len(stage5_reporting_entity_scope)} perimeter rules", required_treatment="Do not attribute consolidated results to brands without support.")
add_check("S5D014", "wings_disclosure", "Private-company disclosure availability is treated neutrally.", "Missing disclosure is not zero or weak performance", "passed_with_caveat", required_treatment="Report evidence asymmetry separately from performance.")
add_check("S5D015", "mayora_scope", "Le Minerale remains outside the strict-control Mayora primary portfolio.", "extended_group_sensitivity_only", required_treatment="Do not change primary attribution based on observed results.")
add_check("S5D016", "unilever_scope", "Unilever ownership and portfolio credit remain time-varying.", "Observation-period ownership required", required_treatment="Do not carry disposed businesses into later outcomes.")
add_check("S5D017", "source_hierarchy", "Primary filings and official disclosures precede secondary contextual sources.", f"{len(stage5_source_requirements)} source classes", required_treatment="Use secondary sources mainly for corroboration or context.")
add_check("S5D018", "company_claims", "Company-attributed strategy–result explanations remain labelled company-reported.", "company_reported status retained", required_treatment="Do not present management attribution as independent causal proof.")
add_check("S5D019", "attribution_strength", "Attribution classes distinguish direct, segment, company, consolidated, temporal, claimed, and unassessable evidence.", f"{len(stage5_attribution_evidence_rules)} attribution rules", required_treatment="Use the narrowest defensible claim level.")
add_check("S5D020", "linkage_eligibility", "Strategy–result links require pre-specified eligibility gates.", f"{len(stage5_linkage_eligibility_rules)} linkage gates", required_treatment="Do not link actions and outcomes solely because both are available.")
add_check("S5D021", "cross_group_comparability", "Cross-group ranking requires compatible metric, entity, geography, period, unit, and accounting scope.", "within-company baseline; conditional cross-company comparison", "passed_with_caveat", required_treatment="Report non-comparability as a valid finding.")
add_check("S5D022", "causal_boundary", "Temporal alignment or company attribution is not treated as causal identification.", "No causal upgrade", required_treatment="Use descriptive or associational language unless a defensible causal design is available.")
add_check("S5D023", "missingness", "Unavailable or undisclosed evidence remains missing rather than zero.", "No missing-to-zero rule", required_treatment="Preserve disclosure and measurement gaps.")
add_check("S5D024", "prohibited_synthesis", "No composite score, arbitrary weighting, performance HHI, or strategy score is designed.", "No prohibited synthesis", required_treatment="Keep results dimension-level and case-based.")
add_check("S5D025", "stage_scope", "Stage 5 outputs contain research design metadata only.", "No strategy observations, result estimates, report, or README", required_treatment="Begin acquisition only after this design gate is accepted.")
add_check("S5D026", "stage_gate", "The strategy–result research design is complete enough to begin controlled evidence acquisition.", "PASS_WITH_CAVEAT", "passed_with_caveat", required_treatment="Carry anticipated disclosure asymmetry and comparability limits into acquisition.")

stage5_design_validation = pd.DataFrame(validation_rows)

if (stage5_design_validation["critical_failure"] == "yes").any():
    raise RuntimeError("Stage 5 design contains a critical validation failure.")
if stage5_design_validation.loc[stage5_design_validation["check_id"] == "S5D026", "result"].squeeze() != "PASS_WITH_CAVEAT":
    raise RuntimeError("Stage 5 design gate is not PASS_WITH_CAVEAT.")

print(f"Stage 5 design checks: {len(stage5_design_validation)}")
print(f"Critical failures: {(stage5_design_validation['critical_failure'] == 'yes').sum()}")
print("Stage 5 design gate: PASS_WITH_CAVEAT")


Stage 5 design checks: 26
Critical failures: 0
Stage 5 design gate: PASS_WITH_CAVEAT


## Deterministic Output Writing and Manifest

Write the research-design metadata with stable formatting and record exact row counts and SHA-256 values. The manifest tracks generated metadata but not itself.


In [13]:
OUTPUT_SPECS = [
    ("metadata/stage5_input_lock.csv", stage5_input_lock, "Locked inherited Stage 0–4 input checksums"),
    ("metadata/stage5_research_questions.csv", stage5_research_questions, "Frozen strategy–result research questions"),
    ("metadata/stage5_period_scope.csv", stage5_period_scope, "Primary, lagged, current-snapshot, interim, and contextual period rules"),
    ("metadata/stage5_strategy_taxonomy.csv", stage5_strategy_taxonomy, "Pre-specified strategic-lever taxonomy"),
    ("metadata/stage5_outcome_taxonomy.csv", stage5_outcome_taxonomy, "Separated portfolio, commercial, profitability, capital, operational, and market outcomes"),
    ("metadata/stage5_reporting_entity_scope.csv", stage5_reporting_entity_scope, "Reporting-entity and portfolio-perimeter rules"),
    ("metadata/stage5_source_requirements.csv", stage5_source_requirements, "Source hierarchy and acquisition requirements"),
    ("metadata/stage5_attribution_evidence_rules.csv", stage5_attribution_evidence_rules, "Attribution classes and evidence-strength rules"),
    ("metadata/stage5_linkage_eligibility_rules.csv", stage5_linkage_eligibility_rules, "Pre-specified strategy–result linkage gates"),
    ("metadata/stage5_design_validation.csv", stage5_design_validation, "Stage 5 research-design validation and final gate"),
]

for relative_path, dataframe, _ in OUTPUT_SPECS:
    destination = OUTPUT_ROOT / relative_path
    destination.parent.mkdir(parents=True, exist_ok=True)
    dataframe.to_csv(destination, index=False, lineterminator="\n")

manifest_rows = []
for relative_path, dataframe, description in OUTPUT_SPECS:
    destination = OUTPUT_ROOT / relative_path
    manifest_rows.append(
        {
            "file_path": relative_path,
            "artifact_type": "csv",
            "row_count": len(dataframe),
            "sha256": sha256_file(destination),
            "description": description,
            "locked_input_commit": INPUT_COMMIT,
        }
    )

stage5_output_manifest = pd.DataFrame(manifest_rows)
manifest_path = OUTPUT_ROOT / "metadata/stage5_output_manifest.csv"
stage5_output_manifest.to_csv(manifest_path, index=False, lineterminator="\n")

if len(stage5_output_manifest) != len(OUTPUT_SPECS):
    raise RuntimeError("Stage 5 output manifest is incomplete.")

print(f"Generated tracked artifacts: {len(stage5_output_manifest)}")
print(f"Manifest: {manifest_path}")


Generated tracked artifacts: 10
Manifest: /content/fmcg_stage5_outputs/metadata/stage5_output_manifest.csv


## Stage 5 Design Preview

Display the frozen design cardinalities, reporting perimeter, evidence-status rules, validation result, and output manifest for review before evidence acquisition begins.


In [14]:
from IPython.display import Markdown, display

display(Markdown("### Design Cardinalities"))
display(
    pd.DataFrame(
        [
            ("Research questions", len(stage5_research_questions)),
            ("Period rules", len(stage5_period_scope)),
            ("Strategic-lever dimensions", len(stage5_strategy_taxonomy)),
            ("Outcome definitions", len(stage5_outcome_taxonomy)),
            ("Reporting-perimeter rules", len(stage5_reporting_entity_scope)),
            ("Source requirement classes", len(stage5_source_requirements)),
            ("Attribution rules", len(stage5_attribution_evidence_rules)),
            ("Linkage gates", len(stage5_linkage_eligibility_rules)),
            ("Validation checks", len(stage5_design_validation)),
            ("Manifest-tracked artifacts", len(stage5_output_manifest)),
        ],
        columns=["component", "count"],
    )
)

display(Markdown("### Reporting-Entity Scope"))
display(stage5_reporting_entity_scope)

display(Markdown("### Attribution and Evidence Strength"))
display(stage5_attribution_evidence_rules)

display(Markdown("### Final Validation Gate"))
display(stage5_design_validation.tail(5))

display(Markdown("### Output Manifest"))
display(stage5_output_manifest)


### Design Cardinalities

,component,count
0,Research questions,11
1,Period rules,7
2,Strategic-lever dimensions,9
3,Outcome definitions,18
4,Reporting-perimeter rules,10
5,Source requirement classes,11
6,Attribution rules,9
7,Linkage gates,16
8,Validation checks,26
9,Manifest-tracked artifacts,10


### Reporting-Entity Scope

,entity_scope_id,canonical_group,reporting_perimeter,portfolio_role,preferred_attribution_unit,required_treatment,prohibited_use
0,ENT01,Wings Group,Wings Group strict-control consumer portfolio,primary_portfolio,group_or_resolved_operating_entity,Group attribution is acceptable; exact operating entity must be resolved when a corporate result requires it.,Do not treat unavailable private-company disclosure as zero or weak performance.
1,ENT02,Wings Group,PT Lion Wings; PT Glico Wings; PT Calbee Wings and other explicit joint ventures,non_primary,joint_venture,Retain separately and do not give full strict-control credit.,JV results cannot be merged into the primary Wings portfolio without ownership-proportion and scope support.
2,ENT03,Indofood,Consumer Branded Products,primary_portfolio,segment_or_controlled_entity,"Use brand, category, segment, or controlled-entity outcomes when the reporting perimeter is explicit.",Do not attribute total Indofood consolidated outcomes to packaged FMCG brands.
3,ENT04,Indofood,Bogasari consumer flour portfolio,primary_portfolio,segment,Include consumer-facing packaged flour results when defensibly attributable.,Exclude industrial-only or unrelated segment outcomes from brand-performance interpretation.
4,ENT05,Indofood,Agribusiness consumer edible oils and fats,primary_portfolio,segment,Include only consumer edible-oil and fat outcomes with adequate product-scope support.,Do not use upstream plantation results as direct packaged-FMCG outcomes.
5,ENT06,Indofood,Distribution and upstream-only operations,context_only,segment_or_group,Use only to explain route-to-market or input context when explicitly linked.,Do not award portfolio-performance credit.
6,ENT07,Mayora,PT Mayora Indah Tbk and consolidated subsidiaries,primary_portfolio,consolidated_or_segment,Preserve domestic/export and segment scope where disclosed.,Do not assume consolidated or export-heavy outcomes represent Indonesian household demand.
7,ENT08,Mayora,PT Tirta Fresindo Jaya / Le Minerale,sensitivity_only,related_group_affiliate,Retain only in the pre-specified extended-group sensitivity.,Do not include in the strict-control primary Mayora portfolio.
8,ENT09,Unilever Indonesia,PT Unilever Indonesia Tbk controlled portfolio,primary_time_varying,company_brand_or_category,Credit outcomes only while the brand or business was controlled during the observation period.,Do not carry disposed businesses into later periods.
9,ENT10,Unilever Indonesia,"Foodservice-only Knorr, Hellmann’s, and Lipton evidence",context_only,brand_or_channel,Use only as channel context outside the primary household-packaged-FMCG universe.,Do not merge foodservice outcomes into the primary portfolio analysis.


### Attribution and Evidence Strength

,attribution_rule_id,attribution_class,definition,default_evidence_status,allowed_claim
0,ATTR01,brand_category_direct,"The action and result match the same controlled brand or exact category, geography, and eligible period.",directly_supported,May support a direct descriptive link; causation still requires stronger identification.
1,ATTR02,business_segment_direct,"The action and result match the same disclosed business segment and eligible period, but brand-level attribution is ...",supported_with_caveat,Report at segment level only.
2,ATTR03,indonesia_company_direct,The result applies to the Indonesian operating company but combines multiple categories or brands.,supported_with_caveat,Do not assign the result to an individual brand or category.
3,ATTR04,consolidated_group_only,"The result combines multiple entities, markets, or businesses beyond the primary analytical universe.",context_only,Do not treat the result as Indonesian packaged-FMCG performance.
4,ATTR05,mixed_geography,Domestic and export or international results cannot be separated.,context_only,Do not interpret the measure as Indonesian household demand.
5,ATTR06,temporally_aligned_only,"A documented action precedes an outcome, but mechanism, entity, category, or alternative explanations remain insuffi...",temporally_aligned,Describe alignment only; do not state that the action produced the result.
6,ATTR07,company_claimed_link,The company explicitly attributes a result to a strategy or operating factor.,company_reported,Label the attribution as company-reported and seek independent or accounting support.
7,ATTR08,plausible_unverified,"A relationship is plausible but lacks adequate direct, temporal, or scope evidence.",plausible_but_unverified,"Retain as interpretation or hypothesis, not a finding."
8,ATTR09,not_attributable,"The action and outcome cannot be matched on entity, geography, category, ownership, or period.",not_assessable,Do not create a strategy–result link.


### Final Validation Gate

,check_id,validation_area,check_description,result,status,critical_failure,required_treatment
21,S5D022,causal_boundary,Temporal alignment or company attribution is not treated as causal identification.,No causal upgrade,passed,no,Use descriptive or associational language unless a defensible causal design is available.
22,S5D023,missingness,Unavailable or undisclosed evidence remains missing rather than zero.,No missing-to-zero rule,passed,no,Preserve disclosure and measurement gaps.
23,S5D024,prohibited_synthesis,"No composite score, arbitrary weighting, performance HHI, or strategy score is designed.",No prohibited synthesis,passed,no,Keep results dimension-level and case-based.
24,S5D025,stage_scope,Stage 5 outputs contain research design metadata only.,"No strategy observations, result estimates, report, or README",passed,no,Begin acquisition only after this design gate is accepted.
25,S5D026,stage_gate,The strategy–result research design is complete enough to begin controlled evidence acquisition.,PASS_WITH_CAVEAT,passed_with_caveat,no,Carry anticipated disclosure asymmetry and comparability limits into acquisition.


### Output Manifest

,file_path,artifact_type,row_count,sha256,description,locked_input_commit
0,metadata/stage5_input_lock.csv,csv,7,d818327e572dc52c4f74db9231f9207383a39a926e425ba13d25919341ce58da,Locked inherited Stage 0–4 input checksums,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
1,metadata/stage5_research_questions.csv,csv,11,6f2854d93713534e89f64ff743d4ec2a0bdd101e9a109989cb13bfcecd1aaa14,Frozen strategy–result research questions,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
2,metadata/stage5_period_scope.csv,csv,7,7c70d6e7c6b1f917ad3685c08f5f0b2d215e7ac62b008b5e7cee0bfec0f3b713,"Primary, lagged, current-snapshot, interim, and contextual period rules",888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
3,metadata/stage5_strategy_taxonomy.csv,csv,9,65c5ffd83c847b9f93c64a4bafd8ca6bf27abe26ee5ea2acd2ff69e07862c684,Pre-specified strategic-lever taxonomy,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
4,metadata/stage5_outcome_taxonomy.csv,csv,18,32a3b729ea1e903a24a30e5927e47e909745e527c42281dd0997a7919320cc03,"Separated portfolio, commercial, profitability, capital, operational, and market outcomes",888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
5,metadata/stage5_reporting_entity_scope.csv,csv,10,6db1c9e177363b9e789be2094d1f922a1a9fcfb7b4d659fe1c3ee71a1be508e8,Reporting-entity and portfolio-perimeter rules,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
6,metadata/stage5_source_requirements.csv,csv,11,bc644cc6dc5fa6ad5fa8a7df7799e1ff5a4512e4beeb0baff08fd5ac3c053c2c,Source hierarchy and acquisition requirements,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
7,metadata/stage5_attribution_evidence_rules.csv,csv,9,a2afb7e170f71d6bb6937cfc41c3e84c027c4e2df82525096a2cf776215006a5,Attribution classes and evidence-strength rules,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
8,metadata/stage5_linkage_eligibility_rules.csv,csv,16,39142fd09351f473c033a8862678e5ff1327871407e05a8b1e6b9093c74b092a,Pre-specified strategy–result linkage gates,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
9,metadata/stage5_design_validation.csv,csv,26,bdff4b05d8f23a1f5f53b8cfd31cfb8f218d3e831a7c70f375403bdbc9be79e0,Stage 5 research-design validation and final gate,888c42ff5a9bab3b8d86eef5c5026eb3203b20bc
